In [1]:
# 这个代码干的事情其实就是用梯度提升回归树 (GradientBoostingRegressor, GBDT) 对地理网格数据建模，并做特征重要性分析 + 偏依赖图 (PDP) 提取。
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedKFold
from scipy.stats import randint, uniform, loguniform

# 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
def run_gbdt(grid_folder, years=[2016, 2023], n=0):
    for year in years:
        target_vars = [f'hr_{year}']
        explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M','X','Y'] # 顺序很讲究
        # 保存结果
        all_results = []
        pdp_records = []
        r2_comparison = []

        param_dist = {
            'n_estimators': [4168], #4168
            'learning_rate': loguniform(0.002, 0.355), #(0.002, 0.355)
            'subsample': uniform(0.545, 0.413), # [0.545,0.958]
            'max_depth' : randint(5, 14), # [5, 13]
            'min_samples_split':[2], #2
            'max_features': uniform(0.335, 0.581), #[0.335,0.916]
            }

        # === 主循环 ===
        for filename in os.listdir(grid_folder):
            if filename.endswith(f'city{year}_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
                input_path = os.path.join(grid_folder, filename)
                match = re.search(r'(\d{3,5})m', filename)
                grid_size = match.group(1)

                gdf = gpd.read_file(input_path)
                gdf_clean = gdf.replace([np.inf, -np.inf], np.nan).dropna(subset=target_vars + explanatory_vars)

                for target in target_vars:
                    X = gdf_clean[explanatory_vars]
                    y = gdf_clean[target]

                    #####################################################################
                    # 用20个 random seeds
                    np.random.seed(0)  # 固定种子以便复现
                    random_seeds = np.random.choice(10000, size=20, replace=False)
                    n = n
                    #####################################################################
                    for r in [random_seeds[n]]:

                        # 数据划分
                        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=r) # 切20次

                        gbdt = GradientBoostingRegressor(random_state=0)
                        
                        cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=0)

                        search = RandomizedSearchCV(
                            estimator=gbdt,
                            param_distributions=param_dist,
                            n_iter= 200,
                            scoring='r2',
                            cv=cv, # cross validation
                            verbose=3,
                            n_jobs=-1,
                            random_state=0
                        )

                        search.fit(X_train, y_train)
                        folder = os.path.join(grid_folder, r'Machine Learning')
                        os.makedirs(folder, exist_ok=True)

                        # 先保存一份 checkpoint (CSV，追加方式)
                        checkpoint_path = os.path.join(folder, f"{year}_{target}_seed{n}_checkpoint_120m.csv")
                        df_cv = pd.DataFrame(search.cv_results_)
                        if not os.path.exists(checkpoint_path):
                            df_cv.to_csv(checkpoint_path, index=False)
                        else:
                            df_cv.to_csv(checkpoint_path, mode='a', header=False, index=False)
                        print(f"[SAVE] checkpoint -> {checkpoint_path}")

                        # 再保存原本的 Excel（完整結果）
                        cv_path = os.path.join(folder, f"{n}_{r}_GBDT_{target}_cv_results_120m.xlsx")
                        try:
                            df_cv.to_excel(cv_path, index=False)
                            print(f"[SAVE] CV results -> {cv_path}")
                        except Exception as e:
                            print(f"[ERROR] Save CV results failed: {cv_path} | {e}")
                            raise

                        # 使用测试集评估
                        best_model = search.best_estimator_
                        y_train_pred = best_model.predict(X_train)
                        y_test_pred = best_model.predict(X_test)

                        r2_train = best_model.score(X_train, y_train)
                        r2_test = r2_score(y_test, y_test_pred)

                        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
                        rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

                        print(f" {filename} | {target} 最佳参数: {search.best_params_} | R²_train={r2_train:.3f} | R²_test={r2_test:.3f}")

                        for var, importance in zip(explanatory_vars, best_model.feature_importances_):
                            all_results.append({
                                'GridSize': grid_size,
                                'Target': target,
                                'Feature': var,
                                'Random seed':r,
                                'FeatureImportance_TrainModel': round(importance, 4),
                                'Train_R2': round(r2_train, 4),
                                'Train_RMSE': round(rmse_train, 4),
                                'Test_R2': round(r2_test, 4),
                                'Test_RMSE': round(rmse_test, 4),
                                **search.best_params_
                            })
                            r2_comparison.append({
                                'GridSize': grid_size,
                                'Target': target,
                                'Random seed':r,
                                'Train_R2': round(r2_train, 4),
                                'Test_R2': round(r2_test, 4),
                                'Train_RMSE': round(rmse_train, 4),
                                'Test_RMSE': round(rmse_test, 4)
                            })
                

                
                #   保存模型训练后的结果
                df_all = pd.DataFrame(all_results)

                df_all.to_excel(os.path.join(folder, f'{year} GBDT_Random_Search_Results_{n}_{r}_120m_hr.xlsx'), index=False)
                df_r2 = pd.DataFrame(r2_comparison)
                df_r2 = df_r2.sort_values(['Target', 'GridSize'])
                df_r2.to_excel(os.path.join(folder, f'{year} R2_Comparison_Train_vs_Test_{n}_{r}_120m_hr.xlsx'), index=False)
                # print("✅ R² train vs test comparison saved.")

run_gbdt(grid_folder, years=[2016], n=0)

Fitting 100 folds for each of 200 candidates, totalling 20000 fits
[SAVE] checkpoint -> D:\seoul\grids\lst_map\final_clean\480_based\Machine Learning\2016_hr_2016_seed0_checkpoint_120m.csv
[SAVE] CV results -> D:\seoul\grids\lst_map\final_clean\480_based\Machine Learning\0_9394_GBDT_hr_2016_cv_results_120m.xlsx
 city2016_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp | hr_2016 最佳参数: {'learning_rate': 0.023732950058719497, 'max_depth': 8, 'max_features': 0.47351811621687767, 'min_samples_split': 2, 'n_estimators': 4168, 'subsample': 0.7574937629823866} | R²_train=0.999 | R²_test=0.959


# SDEM result

In [2]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
from spreg import ML_Error, ML_Lag
from libpysal.weights import lag_spatial

# -----------------------------
# 设置路径
# -----------------------------
grid_folder = r"D:\seoul\grids\lst_map\final_clean\480_based"
output_folder = os.path.join(grid_folder, 'statistics')
os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# 准备变量
# -----------------------------
results_list = []
def show_formula(model, model_type='SDM'):
    # 处理 y 名称
    y_name = model.name_y if isinstance(model.name_y, str) else model.name_y[0]

    coefs = model.betas.flatten()
    vars_ = model.name_x

    terms = []
    for coef, var in zip(coefs, vars_):
        if var.lower() in ['const', 'constant']:  # 常数项
            terms.append(f"{coef:.4f}")
        else:
            terms.append(f"{coef:.4f}*{var}")

    formula = f"{y_name} = "

    # SDM rho 处理
    if model_type == 'SDM' and hasattr(model, 'rho'):
        rho_term = f"{model.rho:.4f}*W{y_name}"
        formula += " + ".join(terms[:1] + [rho_term] + terms[1:])  # 常数项 0 + Wy + X + WX
        return formula

    # SDEM lambda 处理
    if model_type == 'SDEM' and hasattr(model, 'lambda_'):
        terms.append(f"{model.lambda_:.4f}*error")

    formula += " + ".join(terms)
    return formula


param_list = []

for year in [2023, 2016]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
    explanatory_vars_clean = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR']

    for filename in os.listdir(grid_folder):
        if filename.endswith('.shp') and filename.startswith(f'city{year}_lst_ratio_grid_240m'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path)
            gdf = gdf.replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                # 只保留完整数据
                data = gdf[explanatory_vars + [target]].dropna()
                if data.empty:
                    continue

                data_gdf = gdf.loc[data.index]
                print(f"CRS: {data_gdf.crs}, File: {filename}, Target: {target}")

                # -----------------------------
                # 空间权重矩阵
                # -----------------------------
                threshold = 1000
                w = libpysal.weights.DistanceBand.from_dataframe(data_gdf, threshold=threshold, binary=False)
                w.transform = 'r'  # 行标准化
                w_name = 'W1000'
                ds_name = f'yr{year}'

                # -----------------------------
                # y 与 X
                # -----------------------------
                yi = data[target].values.reshape(-1, 1)
                X_main = data[explanatory_vars].values
                WX_clean = lag_spatial(w, data[explanatory_vars_clean].values)
                X_all = np.hstack([np.ones((len(data), 1)), X_main, WX_clean])
                name_x = ['const'] + explanatory_vars + [f"W_{v}" for v in explanatory_vars_clean]
                # print(len(name_x), X_all.shape[1])
                
                n, k = X_main.shape  # 原始自变量数量

                # -----------------------------
                # 模型估计
                # -----------------------------
                models = {}

                # SDM
                models['SDM'] = ML_Lag(
                    yi, X_all, w=w,
                    name_y=target, name_x=name_x,
                    name_w=w_name, name_ds=ds_name,
                    spat_diag=True, spat_impacts=['full']
                )

                # SDEM
                models['SDEM'] = ML_Error(
                    yi, X_all, w=w,
                    name_y=target, name_x=name_x,
                    name_w=w_name, name_ds=ds_name,
                    spat_diag=True
                )

                # -----------------------------
                # 提取 AIC/BIC/logLik
                # -----------------------------
                for model_name, model in models.items():
                    loglik = model.logll
                    aic = model.aic

                    # 参数数量估算
                    if model_name in ['SDM', 'SDEM']:
                        n_params = 2 * k + 3  # k原始 + k滞后 + ρ/λ + 常数 + σ²
                    else:
                        n_params = k + 2

                    bic = -2 * loglik + n_params * np.log(n)

                    results_list.append({
                        "Year": year,
                        "Grid": filename.split("_")[4],
                        "Target": target,
                        "Model": model_name,
                        "AIC": round(aic, 2),
                        "BIC": round(bic, 2),
                        "LogLik": round(loglik, 2),
                        "N": n
                    })

                    formula_sdm = show_formula(models['SDM'], model_type='SDM')
                    print(formula_sdm)
                    formula_sdem = show_formula(models['SDEM'], model_type='SDEM')

                    # 系数转成 dict，列名是变量名
                    coef_sdm = dict(zip(models['SDM'].name_x, models['SDM'].betas.flatten()))
                    coef_sdem = dict(zip(models['SDEM'].name_x, models['SDEM'].betas.flatten()))

                    #  lambda_ 加进去, rho 不用加是因为已经有Wy了
                    if hasattr(models['SDEM'], 'lambda_'):
                        coef_sdem['lambda'] = models['SDEM'].lambda_


                param_list.append({
                    "Year": year,
                    "Grid": filename.split("_")[4],
                    "Target": target,
                    "Model": "SDM",
                    "Formula": formula_sdm,
                    **coef_sdm
                })

                param_list.append({
                    "Year": year,
                    "Grid": filename.split("_")[4],
                    "Target": target,
                    "Model": "SDEM",
                    "Formula": formula_sdem,
                    **coef_sdem
                })


# -----------------------------
# 汇总输出
# -----------------------------
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(["Year", "Target", "Model"])
output_path = os.path.join(output_folder, "SDM_SDEM_clean_AIC_BIC_240m.xlsx")
results_df.to_excel(output_path, index=False)
all_params_df = pd.DataFrame(param_list)
param_output = os.path.join(output_folder, "SDM_SDEM_all_params_240m.xlsx")
all_params_df.to_excel(param_output, index=False)


print(f"✅ 所有模型（SDM/SDEM）AIC/BIC 结果已保存到：\n{output_path}")

CRS: PROJCS["KGD2002_Central_Belt",GEOGCS["GCS_KGD2002",DATUM["D_Korea_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127],PARAMETER["scale_factor",1],PARAMETER["false_easting",200000],PARAMETER["false_northing",500000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]], File: city2023_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp, Target: nor_2023


c:\Users\owner\Python\Python311\Lib\site-packages\scipy\sparse\_data.py:133: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(
c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


nor_2023 = 0.5488 + 0.9464*Wnor_2023 + 0.0279*BCR + -0.0073*BHV + 5.7954*SVF + -24.9575*NDVI + -0.0147*EV + -0.1591*WR + -0.0284*Dist_W + -0.0390*Dist_P + -0.0020*Dist_M + -0.0230*W_BCR + 0.0249*W_BHV + -3.5652*W_SVF + 22.4944*W_NDVI + 0.0149*W_EV + 0.1402*W_WR + 0.9464*W_nor_2023
nor_2023 = 0.5488 + 0.9464*Wnor_2023 + 0.0279*BCR + -0.0073*BHV + 5.7954*SVF + -24.9575*NDVI + -0.0147*EV + -0.1591*WR + -0.0284*Dist_W + -0.0390*Dist_P + -0.0020*Dist_M + -0.0230*W_BCR + 0.0249*W_BHV + -3.5652*W_SVF + 22.4944*W_NDVI + 0.0149*W_EV + 0.1402*W_WR + 0.9464*W_nor_2023
CRS: PROJCS["KGD2002_Central_Belt",GEOGCS["GCS_KGD2002",DATUM["D_Korea_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127],PARAMETER["scale_factor",1],PARAMETER["false_easting",200000],PARAMETER["false_northing",500000],UNIT["metre",1,AUT

c:\Users\owner\Python\Python311\Lib\site-packages\scipy\sparse\_data.py:133: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(
c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


ext_2023 = 0.2838 + 0.9680*Wext_2023 + 0.0274*BCR + -0.0082*BHV + 6.2804*SVF + -26.3818*NDVI + -0.0141*EV + -0.1609*WR + -0.0159*Dist_W + -0.0366*Dist_P + -0.0131*Dist_M + -0.0227*W_BCR + 0.0215*W_BHV + -4.6609*W_SVF + 24.5195*W_NDVI + 0.0144*W_EV + 0.1472*W_WR + 0.9680*W_ext_2023
ext_2023 = 0.2838 + 0.9680*Wext_2023 + 0.0274*BCR + -0.0082*BHV + 6.2804*SVF + -26.3818*NDVI + -0.0141*EV + -0.1609*WR + -0.0159*Dist_W + -0.0366*Dist_P + -0.0131*Dist_M + -0.0227*W_BCR + 0.0215*W_BHV + -4.6609*W_SVF + 24.5195*W_NDVI + 0.0144*W_EV + 0.1472*W_WR + 0.9680*W_ext_2023
CRS: PROJCS["KGD2002_Central_Belt",GEOGCS["GCS_KGD2002",DATUM["D_Korea_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127],PARAMETER["scale_factor",1],PARAMETER["false_easting",200000],PARAMETER["false_northing",500000],UNIT["metre",1,AUT

...

nor_2016 = 1.6464 + 0.9408*Wnor_2016 + 0.0250*BCR + -0.0023*BHV + 6.8634*SVF + -17.0928*NDVI + -0.0140*EV + -0.1006*WR + -0.0085*Dist_W + -0.0254*Dist_P + -0.0053*Dist_M + -0.0336*W_BCR + -0.0005*W_BHV + -5.5349*W_SVF + 12.5746*W_NDVI + 0.0147*W_EV + 0.0824*W_WR + 0.9408*W_nor_2016
nor_2016 = 1.6464 + 0.9408*Wnor_2016 + 0.0250*BCR + -0.0023*BHV + 6.8634*SVF + -17.0928*NDVI + -0.0140*EV + -0.1006*WR + -0.0085*Dist_W + -0.0254*Dist_P + -0.0053*Dist_M + -0.0336*W_BCR + -0.0005*W_BHV + -5.5349*W_SVF + 12.5746*W_NDVI + 0.0147*W_EV + 0.0824*W_WR + 0.9408*W_nor_2016
CRS: PROJCS["KGD2002_Central_Belt",GEOGCS["GCS_KGD2002",DATUM["D_Korea_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127],PARAMETER["scale_factor",1],PARAMETER["false_easting",200000],PARAMETER["false_northing",500000],UNIT["metre",1,A

c:\Users\owner\Python\Python311\Lib\site-packages\scipy\sparse\_data.py:133: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  W.__init__(
c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


ext_2016 = 2.4450 + 0.9494*Wext_2016 + 0.0385*BCR + -0.0069*BHV + 6.9439*SVF + -32.9695*NDVI + -0.0184*EV + -0.1680*WR + -0.0303*Dist_W + -0.0269*Dist_P + -0.0155*Dist_M + -0.0490*W_BCR + 0.0018*W_BHV + -5.9013*W_SVF + 27.4024*W_NDVI + 0.0192*W_EV + 0.1436*W_WR + 0.9494*W_ext_2016
ext_2016 = 2.4450 + 0.9494*Wext_2016 + 0.0385*BCR + -0.0069*BHV + 6.9439*SVF + -32.9695*NDVI + -0.0184*EV + -0.1680*WR + -0.0303*Dist_W + -0.0269*Dist_P + -0.0155*Dist_M + -0.0490*W_BCR + 0.0018*W_BHV + -5.9013*W_SVF + 27.4024*W_NDVI + 0.0192*W_EV + 0.1436*W_WR + 0.9494*W_ext_2016
CRS: PROJCS["KGD2002_Central_Belt",GEOGCS["GCS_KGD2002",DATUM["D_Korea_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127],PARAMETER["scale_factor",1],PARAMETER["false_easting",200000],PARAMETER["false_northing",500000],UNIT["metre",1,AUT

c:\Users\owner\Python\Python311\Lib\site-packages\scipy\sparse\_data.py:133: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  W.__init__(
c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


hr_2016 = -0.6999 + 0.9764*Whr_2016 + -0.0134*BCR + 0.0048*BHV + -0.0644*SVF + 15.8903*NDVI + 0.0044*EV + 0.0669*WR + 0.0178*Dist_W + -0.0008*Dist_P + 0.0074*Dist_M + 0.0165*W_BCR + -0.0021*W_BHV + 0.2586*W_SVF + -14.8523*W_NDVI + -0.0045*W_EV + -0.0610*W_WR + 0.9764*W_hr_2016
hr_2016 = -0.6999 + 0.9764*Whr_2016 + -0.0134*BCR + 0.0048*BHV + -0.0644*SVF + 15.8903*NDVI + 0.0044*EV + 0.0669*WR + 0.0178*Dist_W + -0.0008*Dist_P + 0.0074*Dist_M + 0.0165*W_BCR + -0.0021*W_BHV + 0.2586*W_SVF + -14.8523*W_NDVI + -0.0045*W_EV + -0.0610*W_WR + 0.9764*W_hr_2016
✅ 所有模型（SDM/SDEM）AIC/BIC 结果已保存到：
D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDM_SDEM_clean_AIC_BIC_240m.xlsx
